In [8]:
import time
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm import tqdm

def count_lines(path, buffer_size=1024 * 1024):
    with open(path, "rb") as f:
        return sum(buf.count(b"\n") for buf in iter(lambda: f.read(buffer_size), b""))


def csv_to_daily_spread_cache(csv_path, cache_path, chunksize=2_000_000, sort_values=True):
    """
    Read CSV in chunks and build a pickle cache:
        { Day -> np.array(sorted daily spreads) }

    This is optimized for later pX(pY) calculations.
    """
    csv_path = Path(csv_path)
    cache_path = Path(cache_path)
    tmp_path = cache_path.with_suffix(".tmp.pkl")

    if not csv_path.exists():
        raise FileNotFoundError(f"CSV file not found: {csv_path}")

    tqdm.write("Counting CSV rows...")
    t_count_0 = time.perf_counter()
    total_lines = count_lines(csv_path)
    total_rows = max(total_lines - 1, 0)
    total_chunks = (total_rows + chunksize - 1) // chunksize
    t_count_1 = time.perf_counter()

    tqdm.write(f"Rows: {total_rows:,}")
    tqdm.write(f"Chunks: {total_chunks:,}")
    tqdm.write(f"Line count time: {t_count_1 - t_count_0:.2f} s")

    if cache_path.exists():
        tqdm.write(f"Removing existing cache: {cache_path}")
        cache_path.unlink()

    if tmp_path.exists():
        tqdm.write(f"Removing temp cache: {tmp_path}")
        tmp_path.unlink()

    day_data = {}
    rows_read = 0
    rows_kept = 0
    rows_dropped = 0

    t0 = time.perf_counter()

    reader = pd.read_csv(
        csv_path,
        usecols=["DateTime", "Spread"],
        chunksize=chunksize,
    )

    for chunk in tqdm(reader, total=total_chunks, desc="CSV -> chunk parse", unit="chunk"):
        rows_read += len(chunk)

        chunk["DateTime"] = pd.to_datetime(
            chunk["DateTime"],
            format="%Y%m%d %H:%M:%S",
            errors="coerce"
        )

        chunk["Spread"] = pd.to_numeric(
            chunk["Spread"],
            errors="coerce"
        ).astype("float32")

        before = len(chunk)
        chunk = chunk.dropna(subset=["DateTime", "Spread"])
        rows_dropped += before - len(chunk)

        if chunk.empty:
            continue

        chunk["Day"] = chunk["DateTime"].dt.floor("D")
        rows_kept += len(chunk)

        for day, group in chunk.groupby("Day", sort=False):
            arr = group["Spread"].to_numpy(dtype=np.float32, copy=True)
            day_data.setdefault(day, []).append(arr)

    daily_cache = {}

    for day in tqdm(sorted(day_data.keys()), desc="Merging daily arrays", unit="day"):
        values = np.concatenate(day_data[day]).astype(np.float32, copy=False)
        if sort_values:
            values.sort()
        daily_cache[pd.Timestamp(day)] = values

    tqdm.write("Saving pickle cache...")
    with open(tmp_path, "wb") as f:
        pickle.dump(daily_cache, f, protocol=pickle.HIGHEST_PROTOCOL)

    tmp_path.replace(cache_path)

    t1 = time.perf_counter()

    tqdm.write(f"Rows read: {rows_read:,}")
    tqdm.write(f"Rows kept: {rows_kept:,}")
    tqdm.write(f"Rows dropped: {rows_dropped:,}")
    tqdm.write(f"Days cached: {len(daily_cache):,}")
    tqdm.write(f"Output file: {cache_path}")
    tqdm.write(f"Build time: {t1 - t0:.2f} s")

    return daily_cache


def load_daily_spread_cache(cache_path):
    cache_path = Path(cache_path)
    tqdm.write(f"Loading cache: {cache_path}")
    with open(cache_path, "rb") as f:
        return pickle.load(f)


def compute_spread_px_py_from_cache(daily_cache, daily_percentile, total_percentile):
    """
    Compute pX(pY) from daily cache.

    Example:
        daily_percentile=90, total_percentile=70 -> p70(p90)
    """
    days = sorted(daily_cache.keys())
    daily_vals = np.empty(len(days), dtype=np.float32)

    for i, day in enumerate(tqdm(days, desc=f"Daily p{daily_percentile}", unit="day")):
        values = daily_cache[day]
        daily_vals[i] = np.percentile(values, daily_percentile)

    tqdm.write(f"Computing p{total_percentile}(p{daily_percentile})...")
    final_value = np.percentile(daily_vals, total_percentile)

    daily_df = pd.DataFrame({
        "Day": days,
        f"p{daily_percentile}": daily_vals
    })

    return final_value, daily_df

In [ ]:
# CSV to cache
daily_cache = csv_to_daily_spread_cache(
    csv_path=Path(r"C:\Users\Administrator\Downloads\XAUUSD_dx_darwinex-TICK-No Session.csv"),
    cache_path=Path(r"C:\Users\Administrator\Downloads\XAUUSD_daily_spread_cache.pkl"),
    chunksize=2_000_000,
    sort_values=True
)

In [4]:
# Load cache
daily_cache = load_daily_spread_cache(Path(r"C:\Users\Administrator\Downloads\XAUUSD_daily_spread_cache.pkl"))

Loading cache: C:\Users\Administrator\Downloads\XAUUSD_daily_spread_cache.pkl


In [18]:
value, daily_df = compute_spread_px_py_from_cache(
    daily_cache,
    daily_percentile=90,
    total_percentile=70
)

print(f"p = {value:.5f}")

Daily p90: 100%|████████████████████████████████████████████████████████████████| 2192/2192 [00:01<00:00, 1680.90day/s]

Computing p70(p90)...
p = 0.26000
